# Trash YOLO training

Runs on Colab (GPU). Steps:
1. Clone repo
2. Install deps
3. Load Kaggle credentials
4. Prep TACO dataset
5. Prep Kaggle Drink Waste dataset (optional)
6. Train YOLOv8n
7. Evaluate
8. Save weights to Drive

**Before running:** set `REPO_URL` below to your GitHub URL, and add `KAGGLE_USERNAME` + `KAGGLE_KEY` as Colab secrets (🔑 icon in left sidebar, toggle 'Notebook access').

In [1]:
# --- config ---
REPO_URL = "https://github.com/Ndgandhi23/RU-IEEE-Hackathon.git"
REPO_DIR = "/content/RU-IEEE-Hackathon"
ML_DIR = f"{REPO_DIR}/ml-training"
DRIVE_OUT = "/content/drive/MyDrive/trash-yolo"  # weights will be copied here at the end

In [2]:
# --- mount Drive (for saving weights at the end) ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# --- clone repo ---
import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(ML_DIR)
print('cwd:', os.getcwd())

cwd: /content/RU-IEEE-Hackathon/ml-training


In [ ]:
# --- install deps ---
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 881.3/881.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82

In [ ]:
# --- verify GPU ---
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> GPU, then restart.")

In [ ]:
# --- Kaggle credentials (from Colab secrets) ---
from google.colab import userdata
import os
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
!kaggle --version

In [ ]:
# --- prep TACO (downloads ~1500 images, a few minutes) ---
!python scripts/prepare_dataset.py

In [ ]:
# --- prep Kaggle Drink Waste (~9k images, appends to splits) ---
!python scripts/prepare_drink_waste.py

In [ ]:
# --- class distribution check (after both dataset preps) ---
from collections import Counter
from pathlib import Path

names = ['bottle', 'cup', 'can', 'wrapper', 'paper']
for split in ('train', 'val', 'test'):
    counts = Counter()
    for lbl in Path(f'data/labels/{split}').glob('*.txt'):
        for line in lbl.read_text().strip().splitlines():
            if line:
                counts[int(line.split()[0])] += 1
    total = sum(counts.values())
    print(f"\n{split}: {total} instances")
    for i, n in enumerate(names):
        print(f"  {i} {n}: {counts.get(i, 0)}")

In [ ]:
# --- visual spot-check: render 5 random training images with their boxes ---
import cv2, random
from pathlib import Path
from IPython.display import Image, display

names = ['bottle', 'cup', 'can', 'wrapper', 'paper']
imgs = [p for p in Path('data/images/train').glob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
random.seed(0)
for img_path in random.sample(imgs, min(5, len(imgs))):
    lbl = Path('data/labels/train') / (img_path.stem + '.txt')
    if not lbl.exists() or not lbl.read_text().strip():
        continue
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    for line in lbl.read_text().strip().splitlines():
        cls, cx, cy, bw, bh = map(float, line.split())
        x1, y1 = int((cx - bw/2) * w), int((cy - bh/2) * h)
        x2, y2 = int((cx + bw/2) * w), int((cy + bh/2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, names[int(cls)], (x1, max(0, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    out = f'/tmp/check_{img_path.stem}.jpg'
    cv2.imwrite(out, img)
    display(Image(out, width=480))

In [ ]:
# --- train ---
!python scripts/train.py --epochs 100 --batch 32 --imgsz 960 --name trash_v1


In [ ]:
# --- evaluate on test split ---
!python scripts/evaluate.py runs/train/trash_v1/weights/best.pt --split test

In [ ]:
# --- save weights + run artifacts to Drive ---
import shutil, os
os.makedirs(DRIVE_OUT, exist_ok=True)
src_best = 'runs/train/trash_v1/weights/best.pt'
src_last = 'runs/train/trash_v1/weights/last.pt'
shutil.copy2(src_best, f'{DRIVE_OUT}/trash_v1_best.pt')
shutil.copy2(src_last, f'{DRIVE_OUT}/trash_v1_last.pt')
shutil.copytree('runs/train/trash_v1', f'{DRIVE_OUT}/trash_v1_run', dirs_exist_ok=True)
print('saved to', DRIVE_OUT)